# 资源约束并行机调度问题

**类别：** 调度

来源： [https://www.hexaly.com/templates/resource-constrained-parallel-machine-scheduling](https://www.hexaly.com/templates/resource-constrained-parallel-machine-scheduling)


## 问题

在 Resource-Constrained Parallel Machine Scheduling Problem 中，需要将一组任务分配到若干并行的相同互斥机器上。在同一台机器上，相邻任务之间存在与顺序相关的设置时间（setup times）。每个任务可以被分配可变数量的可再生资源。每个任务的处理时间是非线性的，取决于分配给该任务的资源数量。任何时刻的总资源消耗不得超过全局资源限制。目标是最小化完工时间（makespan），即最后一个任务的完成时间。

	

### 学到的建模原则

- 添加 [interval decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/intervalvariables.html) 来建模任务
- 添加 [list decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模机器上任务的分配及其顺序
- 定义 [lambda functions](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来建模互斥资源和累积资源约束


## 数据

我们提供随机生成的 Resource-Constrained Parallel Machine Scheduling Problem 实例，格式如下：

- 并行机器的数量。
- 任务的数量。
- 每个任务的类型。
- 每个任务的释放日期。
- 执行每个任务所需资源数量的下界。
- 执行每个任务所需资源数量的上界。
- 每个任务的基础持续时间（之后按分配的资源数量归一化）。
- 设置时间矩阵，指示同一台机器上两个相邻任务之间的最小转换时间。


## 程序

Resource-Constrained Parallel Machine Scheduling Problem 的 Hexaly 模型依赖三种类型的决策变量：

- 一个表示机器的 list 决策变量数组：list i 对应于分配到机器 i 的任务序列；
- 一个表示任务时间跨度的 interval 决策变量数组；
- 一个表示每个任务分配资源数量的整数决策变量数组。

我们将问题的约束定义如下。每个任务必须被分配到恰好一台机器，以确保唯一分配。每个任务关联的处理 interval 必须与其所需持续时间匹配，而该持续时间又取决于分配给它的资源数量。

互斥资源约束可以表述如下：对于任意 i，在位置 i+1 处理的任务必须在前一位置 i 上的任务结束后才能开始，二者之间再加上设置时间。为了建模此约束，我们定义了一个 [lambda function](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来表达两个相邻任务之间的关系。然后该函数在每台机器处理的所有任务上被应用于可变参数的 **and** 算子。注意，这些 **and** 表达式中的项数以及 list 的大小（每台机器上分配的任务数）在搜索过程中是变化的。

累积资源约束可以表述如下：对于每个时间槽 t，正在处理的任务所使用的资源数量不得超过总资源数。我们使用可变参数的 **and** 公式结合 [lambda function](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html)，以确保可用资源数量在任何时刻都得到满足。得益于这种可变参数的 **and**，即使时间范围非常大，约束公式仍然紧凑而高效。

目标函数是**最小化完工时间（makespan）**，即最后一个任务的完成时间。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys
import math

def main(instance_file, output_file, time_limit):
    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        with open(instance_file) as f:
            lines = f.readlines()

        # Number of resources
        nb_machines = int(lines[0])

        # Tasks data
        nb_tasks = int(lines[1])
        task_types = [int(i) for i in lines[2].split()]
        release_dates = [int(i) for i in lines[3].split()]

        # Resources data
        nb_resources = int(lines[4])
        min_resources = [int(i) for i in lines[5].split()]
        max_resources = [int(i) for i in lines[6].split()]

        # Durations data
        durations = []
        for line in lines[7:12]:
            line_split = [int(i) for i in line.split()]
            durations.append(line_split)

        # Setup times between two consecutive tasks
        setup = []
        for line in lines[12:]:
            line_split = [int(i) for i in line.split()]
            setup.append(line_split)
        setup = model.array(setup)

        # Horizon: trivial upper bound for the end times of the tasks
        H = max(release_dates) + sum(durations[0])

        # Interval decisions: time range of each task
        tasks = model.array([model.interval(release_dates[i], H) for i in range(nb_tasks)])
        # Number of resources assigned to each task
        nb_assigned_resources = model.array( \
                [model.int(min_resources[task_types[i]], max_resources[task_types[i]]) \
                for i in range(nb_tasks)])

        # List decisions: sequence of tasks on each machine
        tasks_order = model.array([model.list(nb_tasks) for _ in range(nb_machines)])

        # Each task is scheduled on a machine
        model.constraint(model.partition(tasks_order))

        # Non-overlap constraints: the tasks assigned to the same machine are
        # scheduled one after the other, with sequence-dependent setup times
        for m in range(nb_machines):
            sequence = tasks_order[m]
            no_overlap_lambda = model.lambda_function(
                lambda i : model.end(tasks[sequence[i]]) + setup[sequence[i]][sequence[i+1]] \
                        <= model.start(tasks[sequence[i+1]]))
            model.constraint(model.and_( \
                    model.range(0, model.count(sequence)-1), no_overlap_lambda))

        # The task duration depends on the number of assigned resources
        durations = model.array(durations)
        for i in range(nb_tasks):
            model.constraint(model.length(tasks[i]) == durations[nb_assigned_resources[i]-1][i])

        # Makespan: time when all tasks have been processed
        makespan = model.max([model.end(tasks[i]) \
                                    for i in range(nb_tasks)])

        # Cumulative constraint: the number of assigned resources at any time
        # does not exceed the total number of available resources
        capacity_lambda = model.lambda_function(
            lambda t : model.sum(model.contains(tasks[i], t) \
                    * nb_assigned_resources[i] for i in range(0, nb_tasks)) \
                    <= nb_resources
        )
        model.constraint(model.and_(model.range(0, makespan), capacity_lambda))

        # Minimize the makespan
        model.minimize(makespan)

        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = time_limit
        optimizer.solve()

        # Write the solution in a file with the following format:
        # - the total makespan
        # - for each task, the machine, the start and end times,
        #   the number of assigned resources */
        if output_file != None:
            with open(output_file, "w") as f:
                f.write(str(makespan.value))
                f.write("\n")
                for m in range(nb_machines):
                        for i in tasks_order.value[m]:
                            f.write(str(m) + " " \
                                    + str(tasks.value[i].start()) + " " \
                                    + str(tasks.value[i].end()) + " " \
                                    + str(nb_assigned_resources.value[i]))
                            f.write("\n")
                print("Solution written in file ", output_file)


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python resource_constrained_parallel_scheduling.py" \
                "instance_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 60
    main(instance_file, output_file, time_limit)
